In [2]:
import hashlib

MATRICULE = "n04182620182"  # ton matricule

REGIONS = {
    0: ("Afrique de l'Ouest", ["BF", "ML", "NE", "SN", "CI", "GH", "NG", "TG", "BJ", "GN"]),
    1: ("Afrique de l'Est", ["KE", "TZ", "UG", "ET", "RW", "SO", "DJ", "BI"]),
    2: ("Afrique australe", ["ZA", "ZW", "ZM", "BW", "NA", "MZ", "MW"]),
    3: ("Amérique centrale", ["GT", "HN", "SV", "NI", "CR", "PA", "BZ"]),
    4: ("Asie du Sud-Est", ["TH", "VN", "KH", "LA", "MM", "PH", "ID", "MY"]),
    5: ("Europe de l'Est", ["PL", "CZ", "SK", "HU", "RO", "BG", "UA"]),
}

SCENARIOS = {
    0: ("A", "Comité d'investissement régional (ex. BOAD)",
        "Justifier un investissement dans un pays sous-desservi de votre échantillon"),
    1: ("B", "Grand public, via un article de blog",
        "Sensibiliser sur les inégalités d'accès au transport aérien dans votre échantillon"),
    2: ("C", "Compagnie aérienne low-cost",
        "Identifier une opportunité de marché : un pays sans grand aéroport actif"),
}

seed = int(hashlib.sha256(MATRICULE.encode()).hexdigest(), 16)
region_id = seed % len(REGIONS)
scenario_id = (seed // len(REGIONS)) % len(SCENARIOS)

region_name, region_countries = REGIONS[region_id]
scenario_code, scenario_who, scenario_what = SCENARIOS[scenario_id]

print(f"Région assignée   : {region_name} {region_countries}")
print(f"Scénario assigné  : {scenario_code}")
print(f"  Who (audience)  : {scenario_who}")
print(f"  What (message)  : {scenario_what}")


Région assignée   : Afrique de l'Ouest ['BF', 'ML', 'NE', 'SN', 'CI', 'GH', 'NG', 'TG', 'BJ', 'GN']
Scénario assigné  : C
  Who (audience)  : Compagnie aérienne low-cost
  What (message)  : Identifier une opportunité de marché : un pays sans grand aéroport actif


In [3]:
# IMPORT DES DONNEES
import pandas as pd

URL = "https://raw.githubusercontent.com/davidmegginson/ourairports-data/main/airports.csv"

df = pd.read_csv(URL)

print(df.shape)
print(df.head())

(85644, 19)
       id ident           type                  name  latitude_deg  \
0    6523   00A       heliport     Total RF Heliport     40.070985   
1  323361  00AA  small_airport  Aero B Ranch Airport     38.704022   
2    6524  00AK  small_airport          Lowell Field     59.947733   
3    6525  00AL  small_airport          Epps Airpark     34.864799   
4  506791  00AN  small_airport  Katmai Lodge Airport     59.093287   

   longitude_deg  elevation_ft continent iso_country iso_region  municipality  \
0     -74.933689          11.0       NaN          US      US-PA      Bensalem   
1    -101.473911        3435.0       NaN          US      US-KS         Leoti   
2    -151.692524         450.0       NaN          US      US-AK  Anchor Point   
3     -86.770302         820.0       NaN          US      US-AL       Harvest   
4    -156.456699          80.0       NaN          US      US-AK   King Salmon   

  scheduled_service icao_code iata_code gps_code local_code  \
0                

In [4]:
# PRETRAITEMENT DES DONNEES
# A.2 — Prétraitement

# Étape 1 : Garder uniquement les colonnes utiles et les renommer
df_clean = df[["id", "name", "type", "iso_country", "latitude_deg", "longitude_deg", "elevation_ft", "scheduled_service"]].copy()

df_clean.columns = ["ID", "airport_name", "type", "country", "lat", "long", "elevation", "service"]

# Étape 2 : Vérifier les valeurs manquantes
print("Valeurs manquantes par colonne :")
print(df_clean.isnull().sum())

Valeurs manquantes par colonne :
ID                  0
airport_name        0
type                0
country           303
lat                 0
long                0
elevation       14857
service             0
dtype: int64


In [5]:
# Étape 3 : Supprimer les lignes sans pays
df_clean = df_clean.dropna(subset=["country"])
print(f"Lignes après suppression des pays manquants : {len(df_clean)}")

# Étape 4 : Remplacer les élévations manquantes par la médiane
mediane_elevation = df_clean["elevation"].median()
df_clean["elevation"] = df_clean["elevation"].fillna(mediane_elevation)
print(f"Médiane utilisée pour elevation : {mediane_elevation} ft")

# Vérification : plus aucune valeur manquante
print("\nValeurs manquantes après nettoyage :")
print(df_clean.isnull().sum())

# Étape 5 : Sauvegarder
df_clean.to_csv("airports_clean.csv", index=False)
print("\nFichier airports_clean.csv sauvegardé ✅")

Lignes après suppression des pays manquants : 85341
Médiane utilisée pour elevation : 723.0 ft

Valeurs manquantes après nettoyage :
ID              0
airport_name    0
type            0
country         0
lat             0
long            0
elevation       0
service         0
dtype: int64

Fichier airports_clean.csv sauvegardé ✅


In [6]:
# A.3 — Opérations sur la région assignée

# Extraire les aéroports de l'Afrique de l'Ouest
pays_region = ["BF", "ML", "NE", "SN", "CI", "GH", "NG", "TG", "BJ", "GN"]
df_region = df_clean[df_clean["country"].isin(pays_region)]

print(f"Nombre d'aéroports en Afrique de l'Ouest : {len(df_region)}")
print(df_region["country"].value_counts())

Nombre d'aéroports en Afrique de l'Ouest : 301
country
NG    70
BF    51
ML    40
CI    32
NE    28
SN    26
GN    19
GH    18
BJ    10
TG     7
Name: count, dtype: int64


In [7]:
# Filtre 1 : Aéroports dont le nom contient "International" (insensible à la casse)
df_international = df_region[df_region["airport_name"].str.contains("International", case=False)]
print(f"Aéroports 'International' : {len(df_international)}")
print(df_international[["airport_name", "country"]])

# Filtre 2 : Aéroports dont le nom commence par les lettres A à D
df_AD = df_region[df_region["airport_name"].str.startswith(("A", "B", "C", "D"))]
print(f"\nAéroports commençant par A-D : {len(df_AD)}")
print(df_AD[["airport_name", "country"]])

Aéroports 'International' : 37
                                            airport_name country
13071                       Tourou International Airport      BJ
21818            Cotonou Cadjehoun International Airport      BJ
22844   Ouagadougou Thomas Sankara International Airport      BF
22859                       Kotoka International Airport      GH
22861                  Yakubu Tali International Airport      GH
22866                    Prempeh I International Airport      GH
22873       Félix-Houphouët-Boigny International Airport      CI
22896                 Yamoussoukro International Airport      CI
22974       Muhammadu Buhari International Cargo Airport      NG
22982               Nnamdi Azikiwe International Airport      NG
22983                    Akwa Ibom International Airport      NG
22985                Chinua Achebe International Airport      NG
22986                        Asaba International Airport      NG
22988  Sir Abubakar Tafawa Balewa Bauchi State Intern...   

In [8]:
# Aéroport à l'altitude minimale
aeroport_min = df_region.nsmallest(1, "elevation")
print("Aéroport le plus bas :")
print(aeroport_min[["airport_name", "country", "elevation"]])

# Aéroport à l'altitude maximale
aeroport_max = df_region.nlargest(1, "elevation")
print("\nAéroport le plus haut :")
print(aeroport_max[["airport_name", "country", "elevation"]])

Aéroport le plus bas :
              airport_name country  elevation
30230  Saint Louis Airport      SN        9.0

Aéroport le plus haut :
               airport_name country  elevation
23003  Yakubu Gowon Airport      NG     4232.0


In [9]:
# Table de synthèse par pays
synthese = df_region.groupby("country").agg(
    total_aeroports=("airport_name", "count"),
    large_airports=("type", lambda x: (x == "large_airport").sum()),
    service_regulier=("service", lambda x: (x == "yes").sum())
).reset_index()

print(synthese)

# Identifier les pays sous-desservis
# = pays sans aucun grand aéroport EN service régulier
df_region["large_et_service"] = (df_region["type"] == "large_airport") & (df_region["service"] == "yes")

sous_desservis = synthese[synthese["large_airports"] == 0]
print("\nPays sans aucun grand aéroport :")
print(sous_desservis[["country", "total_aeroports", "large_airports", "service_regulier"]])

# Exporter
synthese.to_csv("synthese_pays.csv", index=False)
print("\nFichier synthese_pays.csv sauvegardé ✅")

  country  total_aeroports  large_airports  service_regulier
0      BF               51               2                 2
1      BJ               10               1                 1
2      CI               32               2                 4
3      GH               18               3                 6
4      GN               19               1                 1
5      ML               40               2                 5
6      NE               28               1                 2
7      NG               70              12                27
8      SN               26               2                 4
9      TG                7               2                 1

Pays sans aucun grand aéroport :
Empty DataFrame
Columns: [country, total_aeroports, large_airports, service_regulier]
Index: []

Fichier synthese_pays.csv sauvegardé ✅


/tmp/ipykernel_777/577793040.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_region["large_et_service"] = (df_region["type"] == "large_airport") & (df_region["service"] == "yes")


In [10]:
# Correction du SettingWithCopyWarning
df_region = df_region.copy()

# Redéfinir les pays sous-desservis :
# pays avec peu de service régulier (1 ou moins)
sous_desservis = synthese[synthese["service_regulier"] <= 1]
print("Pays sous-desservis (≤ 1 aéroport en service régulier) :")
print(sous_desservis[["country", "total_aeroports", "large_airports", "service_regulier"]])

Pays sous-desservis (≤ 1 aéroport en service régulier) :
  country  total_aeroports  large_airports  service_regulier
1      BJ               10               1                 1
4      GN               19               1                 1
9      TG                7               2                 1
